In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from tavily import TavilyClient

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage

GEMINI_MODEL = 'gemini-2.5-flash'
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')

model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
    temperature=0.0,
    top_k=1, # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [15]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
ai_msg.content

"J'adore la programmation."

## Abbiamo bisogno di definire uno stato per l'agente che mantiene lo storico delle conversazioni

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [4]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_gemini(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [5]:
from langchain.tools import tool

@tool
def tavily_search_tool(
    query: str, max_results: int = 5
) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        List[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")

    client = TavilyClient(api_key)

    try:
        response = client.search(
            query=query, max_results=max_results
        )

        results = []
        for r in response.get("results", []):
            results.append(
                {
                    "title": r.get("title", ""),
                    "content": r.get("content", ""),
                    "url": r.get("url", ""),
                }
            )

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents

In [6]:
agent = Agent(model=model, system = 'Sei un assistente AI che ha a disposizione dei tool per cercare informazioni online per rispondere alle richieste dell utente. La data corrente è 11/04/2026', tools = [tavily_search_tool])

In [7]:
state = agent.graph.invoke({'messages': [HumanMessage('Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?')]})

Calling: {'name': 'tavily_search_tool', 'args': {'query': 'chi ha vinto il campionato di calcio serie A nel 2025'}, 'id': '0d5b2b16-5820-46c2-ab5b-83a9954fc4d4', 'type': 'tool_call'}
Back to the model!
Calling: {'name': 'tavily_search_tool', 'args': {'query': 'giocatore più pagato del Napoli nel 2025'}, 'id': '1a31f66d-72ed-4d76-9ee0-4c5d1cbc7869', 'type': 'tool_call'}
Back to the model!


In [9]:
state['messages'][-1].text

"Il Napoli ha vinto il campionato di calcio di Serie A nel 2025. Il giocatore più pagato del Napoli in quell'anno è stato Romelu Lukaku, con un ingaggio netto di 6 milioni di euro a stagione."

In [40]:
state['messages']

[HumanMessage(content='Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"max_results": 1, "query": "Chi ha vinto il campionato di calcio serie A nel 2025?"}'}, '__gemini_function_call_thought_signatures__': {'ba9445f8-429d-4e99-9573-403189ced625': 'CsoDAb4+9vtK8uZAoItz9AP/qqu1V7f2XnyDTI57gXLWvMe1p8Vyj0l0JOYU5pkuTU2a1p9ciawQWQs0nd1QXGdo9AIpHFZCvD/U/CwiUbYaFVYHPHZZAc+JcI9qJF40YYyzGi7kcItAaCUNMbj0mWwd7ICu4YEbiP6DBvnkJubJPZw1f5k/p5+zS/oix8CfHva/KMZO7cyolcRTqvgvBi3N2puwXLI+wmXxA9iJTBVxhJwQsSl3OPJ8lbI0oOhXRyNJ5aITdn4nfp8FtwkAedoXImZkcfIDOwmBamh/ROCserulfZ7VLdHwFBqgRLGytCwhqsxq5y+dn3MdqFKlLIaABZZAV1mNYnd6axHKIqJJ+3vkHLmykKu9+tICj52cQSv8KvJ8eTC/T1xQrHhP3orb6rrkZOVFEK/AYYxvFSRMQvEuRQMc/FoIv8YPBIlFyDxkVy/pncYqzs6JF6kKDr72DQELkPd1Y9A0YXq4PReFALg/gPcFLDoYHnsEr3EjrcD055WhUQHegnsiKf0L